In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import datetime

In [14]:
cities = pd.read_csv('..//data//Cities.csv')
display(cities)
concierge = pd.read_csv(
    '..//data//concierge.csv',
    dtype={
        'ID клиента': str,
        'Номер задачи': str,
        'Номер кейса': str
    },
    parse_dates=['Дата и время создания (МСК)'],
    dayfirst=True
 )
display(concierge)

,Unnamed: 0,City,Client_ID
0,0,Amalfi,2
1,1,Grimaud,1
2,2,Phalombe,5
3,3,Абинск,2
4,4,Абу-Даби,3
...,...,...,...
365,365,Южноуральск,1
366,366,Якутск,26
367,367,Ялта,1
368,368,Ярославль,128


,Номер кейса,Номер задачи,ID клиента,Пол,Канал обращения,Ответственный отдел,Категория услуг,Услуга,Программа,Дата и время создания (МСК),Что делали,Страна,Город,"""Технический кейс"""
0,32737,32737.1,19070,NaN,Телефон,Первая Линия,Информационная поддержка,Информационный запрос,Базовый,2024-11-30 21:39:00,"Инфо, Подборка",Россия,Курск,0
1,32646,32646.1,19067,NaN,Мессенджер,Лайфстайл,Товары,Цветы,Базовый,2024-11-30 15:25:00,NaN,Россия,Воронеж,0
2,32583,32583.1,19066,NaN,Мессенджер,Первая Линия,Информационная поддержка,Консьерж программа,Базовый,2024-11-30 07:15:00,Инфо,Россия,Москва,1
3,32571,32571.1,19065,NaN,Мессенджер,Первая Линия,Сервисы,Бесплатная юридическая консультация,Базовый,2024-11-29 23:15:00,Передано партнеру,Россия,Владимир,0
4,32572,32572.1,19065,NaN,Мессенджер,Первая Линия,Информационная поддержка,Информационный запрос,Базовый,2024-11-29 23:30:00,Инфо,Россия,Владимир,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33108,922,NaN,NaN,NaN,NaN,NaN,Путешествия,Авиа,NaN,NaT,NaN,Турция,Анталия,0
33109,921,NaN,NaN,NaN,NaN,NaN,Размещение,Отели,NaN,NaT,NaN,Турция,Анталия,0
33110,920,NaN,NaN,NaN,NaN,NaN,Размещение,Отели,NaN,NaT,NaN,Турция,Бодрум,0
33111,764,NaN,NaN,NaN,NaN,NaN,Мероприятия,Мероприятия,NaN,NaT,NaN,Италия,Лаятико,0


In [15]:
concierge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33113 entries, 0 to 33112
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Номер кейса                  33113 non-null  object        
 1   Номер задачи                 32728 non-null  object        
 2   ID клиента                   32728 non-null  object        
 3   Пол                          11522 non-null  object        
 4   Канал обращения              31681 non-null  object        
 5   Ответственный отдел          31681 non-null  object        
 6   Категория услуг              33113 non-null  object        
 7   Услуга                       33113 non-null  object        
 8   Программа                    29806 non-null  object        
 9   Дата и время создания (МСК)  32728 non-null  datetime64[ns]
 10  Что делали                   30456 non-null  object        
 11  Страна                       32765 non-nu

In [16]:
concierge.describe()

,Дата и время создания (МСК),"""Технический кейс"""
count,32728,33113.000000
mean,2024-09-07 04:27:26.921902080,0.079606
min,2024-04-01 12:14:00,0.000000
25%,2024-07-29 14:17:45,0.000000
50%,2024-09-12 20:37:00,0.000000
75%,2024-10-24 07:37:30,0.000000
max,2024-11-30 23:13:00,1.000000
std,NaN,0.270687


In [21]:
display(concierge['ID клиента'].nunique())
display(concierge['ID клиента'].isna().sum())

5516

np.int64(385)

In [24]:
display(concierge['Номер кейса'].nunique())
display(concierge['Номер кейса'].isna().sum())

24622

np.int64(0)

In [25]:
display(concierge.duplicated().sum())
display(concierge['Номер кейса'].duplicated().sum())
display(concierge['ID клиента'].duplicated().sum())

np.int64(0)

np.int64(8491)

np.int64(27596)

In [36]:
df_us = pd.DataFrame(concierge['Услуга'].unique(), columns=['Услуга'])
display(df_us.sort_values(by='Услуга').reset_index(drop=True).head(40))

,Услуга
0,Авиа
1,Авто
2,Автобусы
3,Алкоголь
4,Аренда транспорта без водителя
5,Аренда транспорта с водителем
6,Афиша
7,Базы отдыха
8,Банкет
9,Бесплатная медицинская консультация


In [42]:
# 1) Впиши сюда 3 ID клиентов, которых считаем высокомаржинальными
high_value_client_ids = ['112', '475', '190']  # TODO: замени на свои ID

# Базовая подготовка
df = concierge.copy()
df['ID клиента'] = df['ID клиента'].astype('string').str.strip()
df = df[df['ID клиента'].notna() & (df['ID клиента'] != '')].copy()

# Флаг информационного запроса
is_info = (
    df['Услуга'].fillna('').str.strip().eq('Информационный запрос')
    | df['Категория услуг'].fillna('').str.contains('Информац', case=False, na=False)
    | df['Что делали'].fillna('').str.contains('Инфо', case=False, na=False)
)
df['is_info'] = is_info.astype(int)

# Отдельные датафреймы
df_top_clients = df[df['ID клиента'].isin(high_value_client_ids)].copy()
df_other_clients = df[~df['ID клиента'].isin(high_value_client_ids)].copy()

print('rows top clients:', len(df_top_clients))
print('rows other clients:', len(df_other_clients))
display(df_top_clients.head(30))

rows top clients: 437
rows other clients: 32291


,Номер кейса,Номер задачи,ID клиента,Пол,Канал обращения,Ответственный отдел,Категория услуг,Услуга,Программа,Дата и время создания (МСК),Что делали,Страна,Город,"""Технический кейс""",is_info
26120,1852,1852.2772,475,NaN,NaN,NaN,Путешествия,ВИП обслуживание в аэропорту,VIP,2024-05-06 16:20:00,NaN,Россия,Москва,0,0
26121,1856,1856.2776,475,NaN,NaN,NaN,Путешествия,ВИП обслуживание в аэропорту,VIP,2024-05-06 19:10:00,NaN,Россия,Москва,0,0
26122,1938,1938.2890,475,NaN,NaN,NaN,Путешествия,ВИП обслуживание в аэропорту,VIP,2024-05-10 16:17:00,NaN,Россия,Москва,0,0
26123,2193,2193.3204,475,NaN,NaN,NaN,Размещение,Отели,VIP,2024-05-16 21:41:00,NaN,Турция,Стамбул,0,0
26124,2193,2193.3270,475,NaN,NaN,NaN,Путешествия,Трансфер,VIP,2024-05-18 10:26:00,NaN,NaN,NaN,0,0
26125,2349,2349.3385,475,NaN,NaN,NaN,Путешествия,ВИП обслуживание в аэропорту,VIP,2024-05-20 20:39:00,NaN,Россия,Москва,0,0
26126,2570,2570.3677,475,NaN,NaN,NaN,Путешествия,ВИП обслуживание в аэропорту,VIP,2024-05-24 15:54:00,NaN,Объединенные Арабские Эмираты,"ОАЭ, Dubai, Дубай",0,0
26127,2598,2598.3706,475,NaN,NaN,NaN,Путешествия,Регистрация на рейс,VIP,2024-05-25 11:09:00,NaN,NaN,NaN,0,0
26128,2631,2631.3745,475,NaN,NaN,NaN,Путешествия,Авиа,VIP,2024-05-27 01:57:00,NaN,NaN,NaN,0,0
26129,2677,2677.3831,475,NaN,NaN,NaN,Путешествия,ВИП обслуживание в аэропорту,VIP,2024-05-28 03:25:00,NaN,Россия,Москва,0,0


In [43]:
def compare_clients(df_all, top_ids):
    data = df_all.copy()
    data['is_top'] = data['ID клиента'].isin(top_ids).astype(int)

    by_client = (
        data.groupby('ID клиента', dropna=False)
        .agg(
            requests=('ID клиента', 'size'),
            info_requests=('is_info', 'sum'),
            unique_services=('Услуга', 'nunique'),
            unique_categories=('Категория услуг', 'nunique'),
            messenger_share=('Канал обращения', lambda s: (s == 'Мессенджер').mean()),
            phone_share=('Канал обращения', lambda s: (s == 'Телефон').mean()),
            top_program_share=('Программа', lambda s: s.fillna('').str.contains('Прем|VIP|Преми|Private', case=False).mean())
        )
        .reset_index()
    )
    by_client['non_info_requests'] = by_client['requests'] - by_client['info_requests']
    by_client['info_share'] = by_client['info_requests'] / by_client['requests']
    by_client['non_info_share'] = 1 - by_client['info_share']
    by_client['is_top'] = by_client['ID клиента'].isin(top_ids).astype(int)

    summary = by_client.groupby('is_top').agg(
        clients=('ID клиента', 'nunique'),
        median_requests=('requests', 'median'),
        median_info_share=('info_share', 'median'),
        median_non_info_share=('non_info_share', 'median'),
        median_unique_services=('unique_services', 'median'),
        median_unique_categories=('unique_categories', 'median')
    )

    corr_cols = [
        'requests', 'info_requests', 'info_share', 'non_info_share',
        'unique_services', 'unique_categories', 'messenger_share', 'phone_share', 'top_program_share', 'is_top'
    ]
    corr = by_client[corr_cols].corr(numeric_only=True)['is_top'].sort_values(ascending=False)

    return by_client, summary, corr

by_client, summary, corr_to_top = compare_clients(df, high_value_client_ids)

print('Сравнение: 0 = остальные, 1 = топ-3')
display(summary)
print('Корреляция признаков с принадлежностью к топ-3:')
display(corr_to_top)

Сравнение: 0 = остальные, 1 = топ-3


,clients,median_requests,median_info_share,median_non_info_share,median_unique_services,median_unique_categories
is_top,,,,,,
0,5513,2.0,0.750000,0.250000,2.0,2.0
1,3,124.0,0.379032,0.620968,22.0,7.0


Корреляция признаков с принадлежностью к топ-3:


is_top               1.000000
requests             0.201035
unique_services      0.138903
info_requests        0.133296
unique_categories    0.067426
top_program_share    0.031387
non_info_share       0.017455
messenger_share      0.012044
info_share          -0.017455
phone_share         -0.017536
Name: is_top, dtype: float64

In [44]:
# Какие инфо-запросы можно убрать первыми (в целом и в топ-3)
info_col = df[df['is_info'] == 1]

automation_candidates = (
    info_col.groupby(['Категория услуг', 'Услуга'], dropna=False)
    .size()
    .reset_index(name='volume')
    .sort_values('volume', ascending=False)
    .head(15)
)

top_info_candidates = (
    df_top_clients[df_top_clients['is_info'] == 1]
    .groupby(['Категория услуг', 'Услуга'], dropna=False)
    .size()
    .reset_index(name='volume_top3')
    .sort_values('volume_top3', ascending=False)
    .head(15)
)

print('Топ-15 инфо-запросов для автоматизации (все клиенты):')
display(automation_candidates)

print('Топ-15 инфо-запросов у топ-3 клиентов:')
display(top_info_candidates)

# Простая оценка эффекта: если автоматизируем X% инфо-запросов
automation_rate = 0.30  # 30%
total_info = int(df['is_info'].sum())
saved_requests = int(total_info * automation_rate)
print(f'Если автоматизировать {automation_rate:.0%} инфо-запросов, можно разгрузить ~{saved_requests} обращений.')

Топ-15 инфо-запросов для автоматизации (все клиенты):


,Категория услуг,Услуга,volume
8,Информационная поддержка,Информационный запрос,2849
9,Информационная поддержка,Консьерж программа,2683
7,Гастро,Ресторан,1764
52,Размещение,Отели,1370
26,Путешествия,Авиа,1209
45,Путешествия,Регистрация на рейс,864
64,Сервисы,Сервисы,580
79,Товары,Товары,517
47,Путешествия,Трансфер,463
23,Мероприятия,Представления,435


Топ-15 инфо-запросов у топ-3 клиентов:


,Категория услуг,Услуга,volume_top3
11,Путешествия,ВИП обслуживание в аэропорту,21
19,Путешествия,Регистрация на рейс,20
4,Информационная поддержка,Информационный запрос,16
10,Путешествия,Авиа,15
32,Товары,Товары,14
20,Путешествия,Трансфер,9
3,Гастро,Ресторан,7
6,Информационная поддержка,Предложения от КС,6
22,Размещение,Отели,6
28,Сервисы,Сервисы,6


Если автоматизировать 30% инфо-запросов, можно разгрузить ~5576 обращений.


In [45]:
# Приоритизация: что автоматизировать первым
all_info = df[df['is_info'] == 1]
top_info = df_top_clients[df_top_clients['is_info'] == 1]

all_by_service = (
    all_info.groupby(['Категория услуг', 'Услуга'], dropna=False)
    .size()
    .rename('volume_all')
    .reset_index()
 )

top_by_service = (
    top_info.groupby(['Категория услуг', 'Услуга'], dropna=False)
    .size()
    .rename('volume_top')
    .reset_index()
 )

priority = all_by_service.merge(top_by_service, on=['Категория услуг', 'Услуга'], how='left')
priority['volume_top'] = priority['volume_top'].fillna(0)
priority['top_share'] = np.where(priority['volume_all'] > 0, priority['volume_top'] / priority['volume_all'], 0)

# Score: большой общий объем + присутствие у топ-клиентов
priority['priority_score'] = priority['volume_all'] * (1 + 2 * priority['top_share'])
priority = priority.sort_values('priority_score', ascending=False).reset_index(drop=True)

display(priority.head(20))

,Категория услуг,Услуга,volume_all,volume_top,top_share,priority_score
0,Информационная поддержка,Информационный запрос,2849,16.0,0.005616,2881.0
1,Информационная поддержка,Консьерж программа,2683,0.0,0.000000,2683.0
2,Гастро,Ресторан,1764,7.0,0.003968,1778.0
3,Размещение,Отели,1370,6.0,0.004380,1382.0
4,Путешествия,Авиа,1209,15.0,0.012407,1239.0
5,Путешествия,Регистрация на рейс,864,20.0,0.023148,904.0
6,Сервисы,Сервисы,580,6.0,0.010345,592.0
7,Товары,Товары,517,14.0,0.027079,545.0
8,Путешествия,Трансфер,463,9.0,0.019438,481.0
9,Мероприятия,Представления,435,2.0,0.004598,439.0
